In [11]:
import pandas as pd
import numpy as np

from pymongo import MongoClient
from dotenv import load_dotenv
import os

In [12]:
load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))

db = client["aqi_predictor"]

collection = db["aqi_features"]

df = pd.DataFrame(list(collection.find()))

In [13]:
if "_id" in df.columns:
    df.drop(columns="_id", inplace=True)

df["timestamp"] = pd.to_datetime(df["timestamp"])

df = df.sort_values("timestamp")

df.reset_index(drop=True, inplace=True)

print(df.shape)

df.head()

(8496, 18)


,city,timestamp,hour,day,month,day_of_week,aqi,aqi_change_rate,pm25,pm10,o3,no2,so2,co,temperature,humidity,pressure,wind_speed
0,karachi,2025-08-05 08:00:00+00:00,8,5,8,1,48,0.0,11.49,45.48,44.97,0.05,0.31,76.93,29.8,70,1002.5,20.9
1,karachi,2025-08-05 09:00:00+00:00,9,5,8,1,47,-1.0,11.23,44.70,44.90,0.05,0.31,76.80,29.5,71,1002.2,20.0
2,karachi,2025-08-05 10:00:00+00:00,10,5,8,1,46,-1.0,10.95,44.54,44.71,0.05,0.31,76.69,29.3,72,1002.0,19.5
3,karachi,2025-08-05 11:00:00+00:00,11,5,8,1,44,-2.0,10.50,44.20,44.39,0.06,0.31,77.06,29.2,72,1001.6,19.9
4,karachi,2025-08-05 12:00:00+00:00,12,5,8,1,43,-1.0,10.21,44.55,43.94,0.06,0.31,77.63,28.9,73,1001.4,19.5


In [14]:
# AQI lag features
df["aqi_lag_1"] = df["aqi"].shift(1)
df["aqi_lag_3"] = df["aqi"].shift(3)
df["aqi_lag_6"] = df["aqi"].shift(6)
df["aqi_lag_12"] = df["aqi"].shift(12)
df["aqi_lag_24"] = df["aqi"].shift(24)

# PM2.5 lag features
df["pm25_lag_1"] = df["pm25"].shift(1)
df["pm25_lag_6"] = df["pm25"].shift(6)
df["pm25_lag_24"] = df["pm25"].shift(24)

df.head(30)

,city,timestamp,hour,day,month,day_of_week,aqi,aqi_change_rate,pm25,pm10,...,pressure,wind_speed,aqi_lag_1,aqi_lag_3,aqi_lag_6,aqi_lag_12,aqi_lag_24,pm25_lag_1,pm25_lag_6,pm25_lag_24
0,karachi,2025-08-05 08:00:00+00:00,8,5,8,1,48,0.0,11.49,45.48,...,1002.5,20.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,karachi,2025-08-05 09:00:00+00:00,9,5,8,1,47,-1.0,11.23,44.70,...,1002.2,20.0,48.0,NaN,NaN,NaN,NaN,11.49,NaN,NaN
2,karachi,2025-08-05 10:00:00+00:00,10,5,8,1,46,-1.0,10.95,44.54,...,1002.0,19.5,47.0,NaN,NaN,NaN,NaN,11.23,NaN,NaN
3,karachi,2025-08-05 11:00:00+00:00,11,5,8,1,44,-2.0,10.50,44.20,...,1001.6,19.9,46.0,48.0,NaN,NaN,NaN,10.95,NaN,NaN
4,karachi,2025-08-05 12:00:00+00:00,12,5,8,1,43,-1.0,10.21,44.55,...,1001.4,19.5,44.0,47.0,NaN,NaN,NaN,10.50,NaN,NaN
5,karachi,2025-08-05 13:00:00+00:00,13,5,8,1,42,-1.0,10.13,42.87,...,1001.6,16.2,43.0,46.0,NaN,NaN,NaN,10.21,NaN,NaN
6,karachi,2025-08-05 14:00:00+00:00,14,5,8,1,43,1.0,10.26,40.95,...,1001.8,14.7,42.0,44.0,48.0,NaN,NaN,10.13,11.49,NaN
7,karachi,2025-08-05 15:00:00+00:00,15,5,8,1,44,1.0,10.64,41.22,...,1002.2,14.5,43.0,43.0,47.0,NaN,NaN,10.26,11.23,NaN
8,karachi,2025-08-05 16:00:00+00:00,16,5,8,1,46,2.0,11.09,44.47,...,1002.8,15.6,44.0,42.0,46.0,NaN,NaN,10.64,10.95,NaN
9,karachi,2025-08-05 17:00:00+00:00,17,5,8,1,48,2.0,11.44,48.09,...,1003.5,14.8,46.0,43.0,44.0,NaN,NaN,11.09,10.50,NaN


In [15]:
# AQI rolling mean
df["aqi_roll_mean_6"] = df["aqi"].rolling(6).mean()

df["aqi_roll_mean_12"] = df["aqi"].rolling(12).mean()

df["aqi_roll_mean_24"] = df["aqi"].rolling(24).mean()

# AQI rolling standard deviation
df["aqi_roll_std_24"] = df["aqi"].rolling(24).std()

df.head(30)

,city,timestamp,hour,day,month,day_of_week,aqi,aqi_change_rate,pm25,pm10,...,aqi_lag_6,aqi_lag_12,aqi_lag_24,pm25_lag_1,pm25_lag_6,pm25_lag_24,aqi_roll_mean_6,aqi_roll_mean_12,aqi_roll_mean_24,aqi_roll_std_24
0,karachi,2025-08-05 08:00:00+00:00,8,5,8,1,48,0.0,11.49,45.48,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,karachi,2025-08-05 09:00:00+00:00,9,5,8,1,47,-1.0,11.23,44.70,...,NaN,NaN,NaN,11.49,NaN,NaN,NaN,NaN,NaN,NaN
2,karachi,2025-08-05 10:00:00+00:00,10,5,8,1,46,-1.0,10.95,44.54,...,NaN,NaN,NaN,11.23,NaN,NaN,NaN,NaN,NaN,NaN
3,karachi,2025-08-05 11:00:00+00:00,11,5,8,1,44,-2.0,10.50,44.20,...,NaN,NaN,NaN,10.95,NaN,NaN,NaN,NaN,NaN,NaN
4,karachi,2025-08-05 12:00:00+00:00,12,5,8,1,43,-1.0,10.21,44.55,...,NaN,NaN,NaN,10.50,NaN,NaN,NaN,NaN,NaN,NaN
5,karachi,2025-08-05 13:00:00+00:00,13,5,8,1,42,-1.0,10.13,42.87,...,NaN,NaN,NaN,10.21,NaN,NaN,45.000000,NaN,NaN,NaN
6,karachi,2025-08-05 14:00:00+00:00,14,5,8,1,43,1.0,10.26,40.95,...,48.0,NaN,NaN,10.13,11.49,NaN,44.166667,NaN,NaN,NaN
7,karachi,2025-08-05 15:00:00+00:00,15,5,8,1,44,1.0,10.64,41.22,...,47.0,NaN,NaN,10.26,11.23,NaN,43.666667,NaN,NaN,NaN
8,karachi,2025-08-05 16:00:00+00:00,16,5,8,1,46,2.0,11.09,44.47,...,46.0,NaN,NaN,10.64,10.95,NaN,43.666667,NaN,NaN,NaN
9,karachi,2025-08-05 17:00:00+00:00,17,5,8,1,48,2.0,11.44,48.09,...,44.0,NaN,NaN,11.09,10.50,NaN,44.333333,NaN,NaN,NaN


In [16]:
FORECAST_HOURS = 72

df["target_aqi"] = [
    df["aqi"].iloc[i + 1:i + 1 + FORECAST_HOURS].mean()
    if i + FORECAST_HOURS < len(df)
    else np.nan
    for i in range(len(df))
]

In [17]:
df = df.dropna()

df.reset_index(drop=True, inplace=True)

print(df.shape)

(8400, 31)


In [18]:
FEATURES = [

    # Calendar
    "hour",
    "day",
    "month",
    "day_of_week",

    # Pollutants
    "pm25",
    "pm10",
    "o3",
    "no2",
    "so2",
    "co",

    # Weather
    "temperature",
    "humidity",
    "pressure",
    "wind_speed",

    # AQI lag
    "aqi_lag_1",
    "aqi_lag_3",
    "aqi_lag_6",
    "aqi_lag_12",
    "aqi_lag_24",

    # PM2.5 lag
    "pm25_lag_1",
    "pm25_lag_6",
    "pm25_lag_24",

    # Rolling
    "aqi_roll_mean_6",
    "aqi_roll_mean_12",
    "aqi_roll_mean_24",
    "aqi_roll_std_24",
]

In [19]:
X = df[FEATURES]

y = df["target_aqi"]

print(X.shape)

print(y.shape)

(8400, 26)
(8400,)


In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    shuffle=False,
)

print("Training")

print(X_train.shape)

print(y_train.shape)

print()

print("Testing")

print(X_test.shape)

print(y_test.shape)

Training
(6720, 26)
(6720,)

Testing
(1680, 26)
(1680,)


In [21]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import numpy as np
import pandas as pd

In [22]:
lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

print("Linear Regression trained successfully.")

Linear Regression trained successfully.


In [23]:
lr_predictions = lr_model.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_predictions)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_predictions))
lr_r2 = r2_score(y_test, lr_predictions)

print(f"MAE  : {lr_mae:.2f}")
print(f"RMSE : {lr_rmse:.2f}")
print(f"R²   : {lr_r2:.4f}")

MAE  : 13.28
RMSE : 16.94
R²   : -0.0933


In [24]:
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [25]:
rf_predictions = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_r2 = r2_score(y_test, rf_predictions)

print(f"MAE  : {rf_mae:.2f}")
print(f"RMSE : {rf_rmse:.2f}")
print(f"R²   : {rf_r2:.4f}")

MAE  : 9.63
RMSE : 12.67
R²   : 0.3885


In [26]:
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest"
    ],
    "MAE": [
        lr_mae,
        rf_mae
    ],
    "RMSE": [
        lr_rmse,
        rf_rmse
    ],
    "R²": [
        lr_r2,
        rf_r2
    ]
})

comparison = comparison.sort_values(
    by="RMSE",
    ascending=True
).reset_index(drop=True)

comparison

,Model,MAE,RMSE,R²
0,Random Forest,9.634611,12.665019,0.388539
1,Linear Regression,13.276649,16.935594,-0.093346


In [27]:
importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": rf_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance

,Feature,Importance
9,co,0.515695
6,o3,0.082756
2,month,0.068373
4,pm25,0.057605
5,pm10,0.051015
1,day,0.048006
24,aqi_roll_mean_24,0.034652
12,pressure,0.023258
3,day_of_week,0.021101
21,pm25_lag_24,0.016754


In [28]:
from sklearn.linear_model import Ridge

In [29]:
ridge_model = Ridge(
    alpha=1.0,
    random_state=42,
)

ridge_model.fit(X_train, y_train)

print("Ridge Regression trained successfully.")

Ridge Regression trained successfully.


In [30]:
ridge_predictions = ridge_model.predict(X_test)

ridge_predictions[:10]

array([60.32474737, 59.09641761, 57.4396565 , 56.5606779 , 55.05928817,
       54.17585893, 53.46948252, 53.78601296, 55.05007855, 56.15188006])

In [31]:
ridge_mae = mean_absolute_error(y_test, ridge_predictions)

ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_predictions))

ridge_r2 = r2_score(y_test, ridge_predictions)

print(f"MAE  : {ridge_mae:.2f}")
print(f"RMSE : {ridge_rmse:.2f}")
print(f"R²   : {ridge_r2:.4f}")

MAE  : 13.28
RMSE : 16.94
R²   : -0.0933


In [32]:
ridge_importance = pd.DataFrame({
    "Feature": FEATURES,
    "Coefficient": ridge_model.coef_
})

ridge_importance["Absolute"] = ridge_importance["Coefficient"].abs()

ridge_importance = ridge_importance.sort_values(
    by="Absolute",
    ascending=False,
)

ridge_importance.drop(columns="Absolute")

,Feature,Coefficient
7,no2,5.448871
4,pm25,1.930174
19,pm25_lag_1,-1.636770
8,so2,-1.629913
2,month,1.615847
12,pressure,1.035093
10,temperature,-0.830617
14,aqi_lag_1,0.760330
13,wind_speed,0.463114
21,pm25_lag_24,-0.444951


In [33]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    ridge_model,
    "../models/ridge_regression.pkl",
)
joblib.dump(lr_model, "../models/linear_regression_lag.pkl")
joblib.dump(rf_model, "../models/random_forest_lag.pkl")

print("Ridge Regression model saved successfully.")

Ridge Regression model saved successfully.


In [34]:
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression",
        "Random Forest",
    ],
    "MAE": [
        lr_mae,
        ridge_mae,
        rf_mae,
    ],
    "RMSE": [
        lr_rmse,
        ridge_rmse,
        rf_rmse,
    ],
    "R²": [
        lr_r2,
        ridge_r2,
        rf_r2,
    ],
})

comparison = comparison.sort_values(
    by="RMSE",
    ascending=True,
).reset_index(drop=True)

comparison

,Model,MAE,RMSE,R²
0,Random Forest,9.634611,12.665019,0.388539
1,Ridge Regression,13.276558,16.935405,-0.093322
2,Linear Regression,13.276649,16.935594,-0.093346


In [35]:
from xgboost import XGBRegressor

In [36]:
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

xgb_model.fit(X_train, y_train)

print("XGBoost trained successfully.")

XGBoost trained successfully.


In [37]:
xgb_predictions = xgb_model.predict(X_test)

xgb_predictions[:10]

array([52.060474, 52.442688, 52.41382 , 52.32174 , 52.506187, 53.950905,
       54.14577 , 55.201515, 55.282352, 53.60365 ], dtype=float32)

In [38]:
xgb_mae = mean_absolute_error(y_test, xgb_predictions)

xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_predictions))

xgb_r2 = r2_score(y_test, xgb_predictions)

print(f"MAE  : {xgb_mae:.2f}")
print(f"RMSE : {xgb_rmse:.2f}")
print(f"R²   : {xgb_r2:.4f}")

MAE  : 8.88
RMSE : 11.85
R²   : 0.4644


In [39]:
xgb_importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": xgb_model.feature_importances_,
})

xgb_importance = xgb_importance.sort_values(
    by="Importance",
    ascending=False,
)

xgb_importance

,Feature,Importance
9,co,0.345610
4,pm25,0.129173
6,o3,0.064467
2,month,0.060582
19,pm25_lag_1,0.057493
22,aqi_roll_mean_6,0.049385
5,pm10,0.044885
24,aqi_roll_mean_24,0.031128
8,so2,0.025460
1,day,0.022259


In [40]:
joblib.dump(
    xgb_model,
    "../models/xgboost.pkl",
)

print("XGBoost model saved successfully.")

XGBoost model saved successfully.


In [41]:
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression",
        "Random Forest",
        "XGBoost",
    ],
    "MAE": [
        lr_mae,
        ridge_mae,
        rf_mae,
        xgb_mae,
    ],
    "RMSE": [
        lr_rmse,
        ridge_rmse,
        rf_rmse,
        xgb_rmse,
    ],
    "R²": [
        lr_r2,
        ridge_r2,
        rf_r2,
        xgb_r2,
    ],
})

comparison = comparison.sort_values(
    by="RMSE",
    ascending=True,
).reset_index(drop=True)

comparison

,Model,MAE,RMSE,R²
0,XGBoost,8.876686,11.853086,0.464426
1,Random Forest,9.634611,12.665019,0.388539
2,Ridge Regression,13.276558,16.935405,-0.093322
3,Linear Regression,13.276649,16.935594,-0.093346


In [42]:
from lightgbm import LGBMRegressor

In [43]:
lgbm_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

lgbm_model.fit(X_train, y_train)

print("LightGBM trained successfully.")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003295 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5281
[LightGBM] [Info] Number of data points in the train set: 6720, number of used features: 26
[LightGBM] [Info] Start training from score 90.045970
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

In [44]:
lgbm_predictions = lgbm_model.predict(X_test)

lgbm_predictions[:10]

array([54.23363341, 54.07207167, 53.26770788, 53.06422684, 53.33742277,
       54.30832762, 53.78285748, 54.26358386, 54.5944777 , 52.50635818])

In [45]:
lgbm_mae = mean_absolute_error(y_test, lgbm_predictions)

lgbm_rmse = np.sqrt(mean_squared_error(y_test, lgbm_predictions))

lgbm_r2 = r2_score(y_test, lgbm_predictions)

print(f"MAE  : {lgbm_mae:.2f}")
print(f"RMSE : {lgbm_rmse:.2f}")
print(f"R²   : {lgbm_r2:.4f}")

MAE  : 9.01
RMSE : 12.17
R²   : 0.4353


In [46]:
lgbm_importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": lgbm_model.feature_importances_,
})

lgbm_importance = lgbm_importance.sort_values(
    by="Importance",
    ascending=False,
)

lgbm_importance

,Feature,Importance
1,day,1057
2,month,561
9,co,503
3,day_of_week,484
6,o3,449
5,pm10,415
8,so2,403
24,aqi_roll_mean_24,391
12,pressure,388
25,aqi_roll_std_24,371


In [47]:
joblib.dump(
    lgbm_model,
    "../models/lightgbm.pkl",
)

print("LightGBM model saved successfully.")

LightGBM model saved successfully.


In [48]:
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression",
        "Random Forest",
        "XGBoost",
        "LightGBM",
    ],
    "MAE": [
        lr_mae,
        ridge_mae,
        rf_mae,
        xgb_mae,
        lgbm_mae,
    ],
    "RMSE": [
        lr_rmse,
        ridge_rmse,
        rf_rmse,
        xgb_rmse,
        lgbm_rmse,
    ],
    "R²": [
        lr_r2,
        ridge_r2,
        rf_r2,
        xgb_r2,
        lgbm_r2,
    ],
})

comparison = comparison.sort_values(
    by="RMSE",
    ascending=True,
).reset_index(drop=True)

comparison

,Model,MAE,RMSE,R²
0,XGBoost,8.876686,11.853086,0.464426
1,LightGBM,9.014308,12.171359,0.435278
2,Random Forest,9.634611,12.665019,0.388539
3,Ridge Regression,13.276558,16.935405,-0.093322
4,Linear Regression,13.276649,16.935594,-0.093346
